# Análise Exploratória: Insurance × Age
Investigando a relação entre a idade e o valor gasto com seguro.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data_3.csv')

# Criar faixas etárias
bins = [18, 25, 35, 45, 55, 64]
labels = ['18-25', '26-35', '36-45', '46-55', '56-64']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)
# Cria uma nova coluna Age_Group e agrupa as idades em 5 faixas etárias para comparações mais limpas

print(f'Shape: {df.shape}')
print(f"\nDistribuição de Age_Group:\n{df['Age_Group'].value_counts().sort_index()}")
print(f"\nEstatísticas de Insurance por Age_Group:")
print(df.groupby('Age_Group', observed=True)['Insurance'].describe().round(2))

Shape: (20000, 28)

Distribuição de Age_Group:
Age_Group
18-25    3034
26-35    4148
36-45    4295
46-55    4224
56-64    3879
Name: count, dtype: int64

Estatísticas de Insurance por Age_Group:
            count     mean      std    min     25%      50%      75%       max
Age_Group                                                                     
18-25      3034.0  1463.46  1652.21  32.20  580.55  1001.26  1788.98  38734.93
26-35      4148.0  1447.55  1509.95  43.82  561.98  1005.11  1748.58  25852.54
36-45      4295.0  1438.11  1379.70  44.35  585.56  1008.50  1805.18  18224.85
46-55      4224.0  1483.86  1516.04  44.72  593.65  1051.32  1830.80  32524.20
56-64      3879.0  1439.47  1441.41  56.67  580.26  1013.62  1770.75  18210.86


## 1. Distribuição de Registros por Faixa Etária

In [ ]:
contagem = df['Age_Group'].value_counts().sort_index().reset_index()
contagem.columns = ['Age_Group', 'Count']
contagem['Age_Group'] = contagem['Age_Group'].astype(str)

fig = px.bar(
    contagem,
    x='Age_Group', y='Count',
    color='Age_Group',
    text='Count',
    title='Quantidade de Registros por Faixa Etária',
    labels={'Age_Group': 'Faixa Etária', 'Count': 'Quantidade'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# Exibição dos números de registros de cada faixa para saber se as faixas
# têm tamanhos parecidos, pois se uma faixa tiver muito mais registros que outra
# qualquer conclusao sobre ela sera mais confiavel, ou o contrário,
# se uma faixa tiver pouquíssimos registros, as análises seguintes sobre ela devem ser interpretadas
# com cautela

## 2. Box Plot — Distribuição do Insurance por Faixa Etária

In [ ]:
fig = px.box(
    df, x='Age_Group', y='Insurance',
    color='Age_Group',
    title='Distribuição do Seguro por Faixa Etária',
    labels={'Insurance': 'Seguro (R$)', 'Age_Group': 'Faixa Etária'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    points='outliers',
    category_orders={'Age_Group': labels}
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# Cada caixa representa uma faixa etária e mostra 5 informações ao mesmo tempo:

# Linha do meio = mediana (valor central da faixa)
# Parte inferior da caixa = Q1 (25% dos dados ficam abaixo)
# Parte superior da caixa = Q3 (75% dos dados ficam abaixo)
# Bigodes = extensão dos valores típicos (1,5x o tamanho da caixa)
# Pontos isolados = outliers, ou seja, valores muito fora do padrão

# Com isso você consegue ver se faixas mais velhas 
# pagam mais seguro, se há mais variabilidade em 
# alguma faixa específica, e se existem 
# casos extremos (pessoas pagando valores absurdamente altos ou baixos).

## 3. Violin Plot — Densidade por Faixa Etária

In [ ]:
fig = px.violin(
    df, x='Age_Group', y='Insurance',
    color='Age_Group',
    box=True, points=False,
    title='Violin Plot — Seguro por Faixa Etária',
    labels={'Insurance': 'Seguro (R$)', 'Age_Group': 'Faixa Etária'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    category_orders={'Age_Group': labels}
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# A largura do violino em cada altura indica quantas pessoas têm aquele valor de seguro
# onde é mais largo, há mais concentração de pessoas
# o mini box plot interno mantém as referências de mediana e quartis.


## 4. Histograma por Faixa Etária

In [ ]:
fig = px.histogram(
    df, x='Insurance', color='Age_Group',
    facet_col='Age_Group',
    nbins=50,
    title='Histograma do Seguro por Faixa Etária',
    labels={'Insurance': 'Seguro (R$)'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    category_orders={'Age_Group': labels}
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# O histograma divide os valores de seguro em 50 intervalos (bins)
# e conta quantas pessoas caem em cada intervalo, gerando um gráfico separado para cada faixa
# o histograma permite ver exatamente onde estão os picos de concentração 

## 5. Estatísticas Resumidas — Média, Mediana e Desvio Padrão

In [ ]:
stats = df.groupby('Age_Group', observed=True)['Insurance'].agg(
    Media='mean', Mediana='median', Desvio_Padrao='std'
).reset_index().round(2)
stats['Age_Group'] = stats['Age_Group'].astype(str)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Média do Seguro', 'Mediana do Seguro', 'Desvio Padrão'])

colors = px.colors.qualitative.Bold[:5]

for i, col in enumerate(['Media', 'Mediana', 'Desvio_Padrao'], 1):
    fig.add_trace(
        go.Bar(x=stats['Age_Group'], y=stats[col],
               marker_color=colors,
               text=stats[col], textposition='outside',
               showlegend=False),
        row=1, col=i
    )

fig.update_layout(title='Estatísticas do Seguro por Faixa Etária', plot_bgcolor='white', height=450)
fig.show()
print(stats.to_string(index=False))

# Três bar charts lado a lado calculados com groupby + agg. A separação entre 
# média e mediana é importante: se a média for muito maior que a mediana, significa que outliers (valores extremos altos) 
# estão puxando a média para cima, e a mediana representa melhor o "usuário típico". O desvio padrão mede o quanto os valores se dispersam em torno 
# da média — um desvio alto indica que pessoas da mesma faixa etária pagam valores muito diferentes entre si, ou seja, a idade sozinha não explica bem o valor do seguro.


Age_Group   Media  Mediana  Desvio_Padrao
    18-25 1463.46  1001.26        1652.21
    26-35 1447.55  1005.11        1509.95
    36-45 1438.11  1008.50        1379.70
    46-55 1483.86  1051.32        1516.04
    56-64 1439.47  1013.62        1441.41


## 6. Média do Seguro por Idade (linha)

In [ ]:
media_por_idade = df.groupby('Age')['Insurance'].mean().reset_index()

fig = px.line(
    media_por_idade,
    x='Age', y='Insurance',
    markers=True,
    title='Média do Seguro por Idade',
    labels={'Age': 'Idade', 'Insurance': 'Média do Seguro (R$)'}
)
fig.update_traces(line_color='#4C72B0', marker_color='#DD8452')
fig.update_layout(plot_bgcolor='white')
fig.show()

# Calcula a média de seguro para cada idade individualmente (18, 19, 20... até 64)
# o resultado é um gráfico de linha que mostra a evolução ano a ano
# Isso revela padrões mais finos, pois anomalias em idades específicas ficam visíveis aqui mas
# seriam mascaradas pelo agrupamento em faixas

## 7. Conclusões

Com base na análise acima:

- O valor gasto com **seguro tende a crescer com a idade**, o que é esperado dado o maior risco de saúde em idades avançadas.
- O **desvio padrão** é alto em todas as faixas, indicando grande variabilidade mesmo dentro da mesma faixa etária.
- O scatter com linha de tendência confirma a **correlação positiva** entre idade e gasto com seguro.
- A faixa **56-64** apresenta a maior média e mediana de seguro.
- A proporção **Insurance/Income** também tende a aumentar com a idade, sugerindo que pessoas mais velhas comprometem uma parcela maior da renda com seguro.